# Sesión 1 — Tu primera llamada a un LLM vía API

**Objetivo:** enviar un mensaje a un LLM desde Python, recibir la respuesta y ver cómo cambia el comportamiento del modelo al darle un rol con un *system prompt*.

El notebook trabaja con **OpenAI** y **Anthropic** a la vez. Si solo tienes la key de uno de ellos, las celdas del otro se saltan con un aviso.

| Nivel | Qué hace | Obligatorio |
|---|---|---|
| 1 | Llamada básica: mensaje → respuesta → `print` | ✅ |
| 2 | Mismo mensaje con un *system prompt* de experto en estimación, y comparación | ✅ |
| 3 | Metadatos (tokens, modelo) y coste estimado | Opcional |

### Cómo ejecutarlo

- **Google Colab:** abre el panel 🔑 *Secrets* y crea `OPENAI_API_KEY` y/o `ANTHROPIC_API_KEY`, con acceso activado para este notebook. Después, *Runtime → Run all*.
- **En local:** `cd session_1 && cp .env.example .env`, rellena las keys, ejecuta `uv sync` y abre el notebook con el kernel del `.venv` (VS Code o Jupyter).

## 0. Setup

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # En Colab no vienen instalados los SDKs; en local los instala `uv sync`.
    get_ipython().run_line_magic("pip", "install -q openai anthropic")
else:
    from dotenv import load_dotenv

    load_dotenv()

print(f"Entorno: {'Google Colab' if IN_COLAB else 'local'}")

Entorno: local


In [2]:
import os


def get_secret(name: str, default: str | None = None) -> str | None:
    """Lee un secreto de Colab Secrets (en Colab) o de las variables de entorno / .env (en local)."""
    value = None
    if IN_COLAB:
        from google.colab import userdata

        try:
            value = userdata.get(name)
        except Exception:  # secreto no creado o sin acceso concedido al notebook
            value = None
    else:
        value = os.getenv(name)
    return value or default


OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY")

OPENAI_MODEL = get_secret("OPENAI_MODEL", "gpt-4o-mini")
# claude-haiku-5-5 es mucho más barato si quieres ahorrar
ANTHROPIC_MODEL = get_secret("ANTHROPIC_MODEL", "claude-opus-5-5")

HAS_OPENAI = bool(OPENAI_API_KEY)
HAS_ANTHROPIC = bool(ANTHROPIC_API_KEY)

print(f"OpenAI:    {'✅ key encontrada' if HAS_OPENAI else '❌ sin key (se saltará)'} · modelo {OPENAI_MODEL}")
print(f"Anthropic: {'✅ key encontrada' if HAS_ANTHROPIC else '❌ sin key (se saltará)'} · modelo {ANTHROPIC_MODEL}")

OpenAI:    ✅ key encontrada · modelo gpt-4o-mini
Anthropic: ✅ key encontrada · modelo claude-opus-5-5


In [3]:
from anthropic import Anthropic
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY) if HAS_OPENAI else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if HAS_ANTHROPIC else None

# Guardamos cada respuesta cruda para reutilizarla en los niveles 2 y 3 sin repetir llamadas.
# Clave: (proveedor, nivel) -> objeto de respuesta del SDK
responses = {}


def anthropic_text(response) -> str:
    """Une los bloques de texto de una respuesta de Anthropic.

    `response.content` es una lista de bloques (text, thinking, tool_use...). Los modelos
    Claude actuales piensan antes de responder, así que puede haber bloques `thinking`
    además del texto: solo nos quedamos con los de tipo `text`.
    """
    return "".join(block.text for block in response.content if block.type == "text")


def skip(provider: str) -> None:
    print(f"⏭️  Sin API key de {provider}: celda saltada.")

## Nivel 1 — Llamada básica

Mandamos un único mensaje con rol `user`, sin ninguna instrucción adicional.

In [ ]:
USER_MESSAGE = (
    "Quiero desarrollar una app móvil para gestionar reservas en restaurantes: registro de "
    "usuarios, búsqueda de restaurantes por tipo de cocina y ubicación, reservas en tiempo real "
    "y notificaciones de confirmación. ¿Cuánto tiempo y dinero costaría?"
)

### OpenAI

`chat.completions.create` recibe una lista de mensajes y devuelve `choices`. El texto está en `choices[0].message.content`.

In [5]:
if HAS_OPENAI:
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": USER_MESSAGE}],
    )
    responses[("openai", 1)] = response
    print(response.choices[0].message.content)
else:
    skip("OpenAI")

El desarrollo de una app móvil para gestionar reservas en restaurantes puede variar en tiempo y costo dependiendo de varios factores, como la complejidad de las funcionalidades, el diseño, la plataforma (iOS, Android o ambas) y si decides contratar a un equipo de desarrollo o realizarlo internamente.

### 1. **Funcionalidades Clave**:
   - Registro de usuarios (inicio de sesión, autenticación)
   - Búsqueda de restaurantes por tipo de cocina y ubicación
   - Sistema de reservas en tiempo real
   - Notificaciones de confirmación (pueden ser push notifications)
   - Posibilidad de calificar y dejar comentarios sobre restaurantes
   - Integración con mapas para ubicación de restaurantes
   - Otras funcionalidades como pasarelas de pago, historial de reservas, etc.

### 2. **Tiempo de Desarrollo**:
   - **Planificación y diseño**: 2-4 semanas
   - **Desarrollo de backend (servidores, bases de datos, API)**: 6-12 semanas
   - **Desarrollo de frontend (app móvil)**: 6-12 semanas
   - **Prueb

### Anthropic

`messages.create` exige `max_tokens`, que es el límite de tokens de salida. El texto llega en bloques dentro de `response.content`.

In [6]:
if HAS_ANTHROPIC:
    response = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=16000,
        messages=[{"role": "user", "content": USER_MESSAGE}],
    )
    responses[("anthropic", 1)] = response
    print(anthropic_text(response))
else:
    skip("Anthropic")

# Estimación: App de reservas para restaurantes

Las cifras son **rangos orientativos**. El coste real depende sobre todo de **quién la desarrolla** (país y tipo de equipo) y del **nivel de complejidad** que quieras.

## Un punto clave antes de empezar

Para que las reservas sean realmente "en tiempo real", los restaurantes necesitan gestionar sus mesas y horarios. En la práctica, esto implica construir **tres productos**:

1. **App móvil para clientes** (iOS + Android)
2. **Panel web para restaurantes** (gestionar mesas, horarios, aceptar o rechazar reservas)
3. **Panel de administración** (para ti: alta de restaurantes, usuarios, incidencias)

Muchos presupuestos olvidan el punto 2, y suele ser de los más costosos.

## Desglose por módulos (MVP)

| Módulo | Horas estimadas |
|---|---|
| Descubrimiento, UX/UI y diseño | 80 – 150 h |
| Registro y login (email, Google, Apple) | 40 – 80 h |
| Búsqueda por cocina y ubicación (mapas, geolocalización, filtros) | 80 – 150 h |
| Fichas de res

👉 Si ves una respuesta sin errores, el setup es correcto.

## Nivel 2 — System prompt con rol

El *system prompt* fija **quién es** el modelo y **cómo debe responder**, antes de que llegue el mensaje del usuario. Enviamos el mismo `USER_MESSAGE` que en el nivel 1.

Diferencia entre SDKs:
- **OpenAI:** el system prompt es un mensaje más, con `role: "system"`, al principio de `messages`.
- **Anthropic:** va en un parámetro aparte, `system=`, y nunca dentro de `messages`.

In [7]:
SYSTEM_PROMPT = """Eres un experto en estimación de proyectos de software con más de 15 años de experiencia en consultoría.

Cuando te pidan estimar un proyecto:
- Desglosa el trabajo en fases (descubrimiento, diseño, desarrollo, QA y despliegue) con las horas de cada fase.
- Indica el total de horas y el coste, usando 62,50 €/h para desarrollo y 50 €/h para diseño.
- Enumera los supuestos que has hecho y los principales riesgos.
- Si falta información relevante, plantéala como pregunta abierta en vez de inventarla.

Responde en español, con un tono profesional y conciso, usando Markdown."""

In [8]:
if HAS_OPENAI:
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_MESSAGE},
        ],
    )
    responses[("openai", 2)] = response
    print(response.choices[0].message.content)
else:
    skip("OpenAI")

### Estimación de Proyecto: App de Gestión de Reservas en Restaurantes

#### Fases del Proyecto

1. **Descubrimiento**
   - Análisis de requerimientos, definición del alcance, entrevistas con stakeholders.
   - **Horas estimadas:** 40 horas
   - **Coste:** 2,000 € (40h * 50 €/h)

2. **Diseño**
   - Diseño de la interfaz de usuario (UI) y experiencia de usuario (UX), prototipos y wireframes.
   - **Horas estimadas:** 60 horas
   - **Coste:** 3,750 € (60h * 62,50 €/h)

3. **Desarrollo**
   - Programación de las funcionalidades: registro de usuarios, búsqueda de restaurantes, gestión de reservas y notificaciones.
   - **Horas estimadas:** 120 horas
   - **Coste:** 7,500 € (120h * 62,50 €/h)

4. **QA (Aseguramiento de Calidad)**
   - Pruebas de funcionalidad, usabilidad y carga, corrección de errores.
   - **Horas estimadas:** 40 horas
   - **Coste:** 2,500 € (40h * 62,50 €/h)

5. **Despliegue**
   - Preparación del entorno de producción, lanzamiento en la tienda de aplicaciones y capacita

In [9]:
if HAS_ANTHROPIC:
    response = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=16000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": USER_MESSAGE}],
    )
    responses[("anthropic", 2)] = response
    print(anthropic_text(response))
else:
    skip("Anthropic")

# Estimación: App móvil de reservas en restaurantes

## Alcance considerado

- **App móvil** (iOS y Android) para comensales: registro e inicio de sesión, búsqueda por tipo de cocina y ubicación, ficha de restaurante, reservas en tiempo real, historial y cancelación, y notificaciones.
- **Backend y API** con base de datos y lógica de disponibilidad.
- **Panel web básico para restaurantes** que permite configurar horarios, capacidad y mesas, y gestionar reservas. Sin él no es posible ofrecer disponibilidad en tiempo real, salvo que exista una integración externa (ver preguntas abiertas).

## Desglose por fases

| Fase | Actividades principales | Horas | Tarifa | Coste |
|---|---|---:|---:|---:|
| **Descubrimiento** | Talleres de requisitos, definición de arquitectura y backlog priorizado | 40 h | 62,50 €/h | 2.500 € |
| **Diseño UX/UI** | Flujos de usuario, *wireframes*, UI de unas 20 pantallas de app y del panel, sistema de diseño y prototipo navegable | 120 h | 50 €/h | 6.000 € |
| **

### Comparación: nivel 1 vs nivel 2

Las dos respuestas de cada proveedor, una al lado de la otra.

In [10]:
import html

from IPython.display import HTML, display


def response_text(provider: str, response) -> str:
    if provider == "openai":
        return response.choices[0].message.content or ""
    return anthropic_text(response)


def show_side_by_side(provider: str) -> None:
    if (provider, 1) not in responses or (provider, 2) not in responses:
        print(f"⏭️  Faltan respuestas de {provider}: no hay nada que comparar.")
        return
    columns = ""
    for level, title in [(1, "Nivel 1 · sin system prompt"), (2, "Nivel 2 · con system prompt")]:
        text = html.escape(response_text(provider, responses[(provider, level)]))
        columns += (
            f"<div style='flex:1;min-width:0'><h4>{title}</h4>"
            f"<pre style='white-space:pre-wrap;font-size:12px'>{text}</pre></div>"
        )
    display(HTML(f"<h3>{provider}</h3><div style='display:flex;gap:16px'>{columns}</div>"))


for provider in ["openai", "anthropic"]:
    show_side_by_side(provider)

In [11]:
import re


def text_metrics(text: str) -> dict:
    """Métricas simples para comparar la forma de dos respuestas."""
    lines = [line.strip() for line in text.splitlines()]
    return {
        "caracteres": len(text),
        "palabras": len(text.split()),
        "líneas": sum(1 for line in lines if line),
        "encabezados": sum(1 for line in lines if line.startswith("#")),
        "items lista": sum(1 for line in lines if re.match(r"([-*]|\d+\.)\s", line)),
        "filas tabla": sum(1 for line in lines if line.startswith("|")),
    }


header = f"{'proveedor':<10} {'nivel':>5} " + " ".join(
    f"{name:>12}" for name in text_metrics("")
)
print(header)
print("-" * len(header))
for (provider, level), response in sorted(responses.items()):
    metrics = text_metrics(response_text(provider, response))
    print(f"{provider:<10} {level:>5} " + " ".join(f"{value:>12}" for value in metrics.values()))

proveedor  nivel   caracteres     palabras       líneas  encabezados  items lista  filas tabla
----------------------------------------------------------------------------------------------
anthropic      1         3963          706           57            8           16           25
anthropic      2         6012          982           67            9           25           29
openai         1         2090          295           24            4           18            0
openai         2         1975          300           35            6           29            0


### 👉 Observaciones

Compara cada par de respuestas y anota lo que ves:

- **Tono:** ¿más genérico o más profesional y de consultor? ¿Más directo?
- **Nivel de detalle:** ¿da cifras concretas (horas, €) o rangos vagos? ¿Explica supuestos y riesgos?
- **Estructura:** ¿texto corrido o secciones, listas y tablas? ¿Sigue el formato que pide el system prompt?
- **Entre proveedores:** ¿qué modelo se ciñe más a las instrucciones?

**Mis conclusiones:**

- _…_

## Nivel 3 (opcional) — Metadatos y coste

Cada respuesta trae el **modelo** que la generó y el **uso de tokens**:

| | Tokens de entrada | Tokens de salida | Modelo |
|---|---|---|---|
| OpenAI | `usage.prompt_tokens` | `usage.completion_tokens` | `model` |
| Anthropic | `usage.input_tokens` | `usage.output_tokens` | `model` |

Coste = `tokens_entrada × precio_entrada + tokens_salida × precio_salida`. Los precios se publican en USD por millón de tokens:

- OpenAI: https://platform.openai.com/pricing
- Anthropic: https://www.anthropic.com/pricing

> En los modelos Claude actuales, `output_tokens` **incluye los tokens de razonamiento (thinking)**. Se facturan aunque el texto de ese razonamiento no se muestre.

In [12]:
# USD por 1M de tokens (entrada, salida). Revísalos en las páginas oficiales: cambian.
PRICES_PER_1M = {
    "gpt-4o-mini": (0.15, 0.60),  # ⚠️ verificar en platform.openai.com/pricing
    "claude-opus-5-5": (4.00, 20.00),  # anthropic.com/pricing (octubre 2026)
    "claude-haiku-5-5": (0.10, 0.50),  # anthropic.com/pricing (octubre 2026), prompts ≤ 100K tokens
}


def find_price(model: str) -> tuple[float, float] | None:
    """Busca el precio del modelo. La API puede devolver un id con fecha
    (p. ej. gpt-4o-mini-2024-07-18), así que también se acepta un prefijo conocido."""
    if model in PRICES_PER_1M:
        return PRICES_PER_1M[model]
    prefixes = [key for key in PRICES_PER_1M if model.startswith(key)]
    return PRICES_PER_1M[max(prefixes, key=len)] if prefixes else None


def estimate_cost(model: str, input_tokens: int, output_tokens: int) -> float | None:
    price = find_price(model)
    if price is None:
        return None
    price_in, price_out = price
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000


def extract_metadata(provider: str, response) -> dict:
    if provider == "openai":
        input_tokens, output_tokens = response.usage.prompt_tokens, response.usage.completion_tokens
    else:
        input_tokens, output_tokens = response.usage.input_tokens, response.usage.output_tokens
    return {
        "model": response.model,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost": estimate_cost(response.model, input_tokens, output_tokens),
    }

In [13]:
# Así es el objeto `usage` crudo de cada proveedor
for provider in ["openai", "anthropic"]:
    response = responses.get((provider, 1))
    if response is not None:
        print(f"{provider}: {response.usage}\n")

openai: CompletionUsage(completion_tokens=494, prompt_tokens=53, total_tokens=547, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))

anthropic: Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=111, output_tokens=2621, output_tokens_details=OutputTokensDetails(thinking_tokens=792), server_tool_use=None, service_tier='standard', speed='standard')



In [14]:
print(f"{'proveedor':<10} {'nivel':>5} {'modelo':<28} {'in':>7} {'out':>7} {'coste USD':>12}")
print("-" * 74)
total = 0.0
for (provider, level), response in sorted(responses.items()):
    meta = extract_metadata(provider, response)
    cost = meta["cost"]
    cost_str = f"{cost:.6f}" if cost is not None else "sin precio"
    total += cost or 0.0
    print(
        f"{provider:<10} {level:>5} {meta['model']:<28} "
        f"{meta['input_tokens']:>7} {meta['output_tokens']:>7} {cost_str:>12}"
    )
print("-" * 74)
print(f"{'TOTAL':<62} {total:>12.6f}")

proveedor  nivel modelo                            in     out    coste USD
--------------------------------------------------------------------------
anthropic      1 claude-opus-5-5                  111    2621     0.052864
anthropic      2 claude-opus-5-5                  342    3917     0.079708
openai         1 gpt-4o-mini-2024-07-18            53     494     0.000304
openai         2 gpt-4o-mini-2024-07-18           195     519     0.000341
--------------------------------------------------------------------------
TOTAL                                                              0.133217


👉 Fíjate en cómo el system prompt aumenta los **tokens de entrada**, porque se envía en cada llamada, y normalmente también los **de salida**, porque pide una respuesta más estructurada y completa. Más calidad suele implicar más coste.